In [ ]:
#MESSY STUDENT DATA
from pyspark.sql import SparkSession, functions as function

# Create SparkSession
spark = SparkSession.builder.appName("StudentDataPreprocessing").getOrCreate()
csv_path = r"sample_data/student_data_part1.csv"

In [ ]:
# Read as DataFrame with explicit DDL schema (schemas + DataFrameReader)
schema_ddl = "Name STRING, Gender STRING, Grade INT, Math DOUBLE, Science DOUBLE, English DOUBLE, Total DOUBLE"
df = spark.read.csv(csv_path, header=True, schema=schema_ddl)

In [ ]:
# TRANSFORM – cleaning and structuring (TRANSFORM step, withColumn, expressions)
# 1) Normalize name (strip quotes/spaces, title case)
df = df.withColumn(
    "name",
    function.initcap(
        function.trim(
            function.regexp_replace(
                function.regexp_replace(function.col("Name"), "'", ""),
                '"',
                ""
            )
        )
    )
)

In [ ]:
# 2) Normalize gender to Male/Female/Unknown (expressions, withColumn)
gender_clean = function.lower(function.trim(function.col("Gender")))
df = df.withColumn(
    "gender_std",
    function.when(gender_clean.isin("m", "male", "1"), function.lit("Male"))
     .when(gender_clean.isin("f", "female", "0"), function.lit("Female"))
     .otherwise(function.lit("Unknown"))
)

In [ ]:
# 3) Normalize grade to integer (expressions on columns)
df = df.withColumn(
    "grade_int",
    function.regexp_extract(function.col("Grade"), r"(\d+)", 1).cast("int")
)

In [ ]:
# 4) Clean subject scores: remove ' marks', cast to double
for old_col, new_col in [("Math", "math_clean"), ("Science", "science_clean"), ("English", "english_clean")]:
    df = df.withColumn(
        new_col,
        function.regexp_replace(function.col(old_col), r"\s*marks", "").cast("double")
    )

In [ ]:
# 5) Recompute total from cleaned subjects (withColumn, expressions)
df = df.withColumn(
    "total_clean",
    function.col("math_clean") + function.col("science_clean") + function.col("english_clean")
)

In [ ]:
# HANDLING MISSING DATA AND DUPLICATES
# Simple rule: drop rows where key fields are null
df = df.filter(
    function.col("math_clean").isNotNull()
    & function.col("science_clean").isNotNull()
    & function.col("english_clean").isNotNull()
    & function.col("grade_int").isNotNull()
    & function.col("name").isNotNull()
)

In [ ]:
# Drop duplicate records based on logical student identity
df = df.dropDuplicates(["name", "gender_std", "grade_int"])

In [ ]:
# Optional: simple mean imputation example
means = df.agg(
    function.mean("math_clean").alias("m_mean"),
    function.mean("science_clean").alias("s_mean"),
    function.mean("english_clean").alias("e_mean")
).collect()[0]
df = df.fillna(
    {
        "math_clean": float(means["m_mean"]),
        "science_clean": float(means["s_mean"]),
        "english_clean": float(means["e_mean"])
    }
).withColumn(
    "total_clean",
    function.col("math_clean") + function.col("science_clean") + function.col("english_clean")
)

In [ ]:
# INSIGHTS (aggregation, groupBy, where)
# Insight 1: average total by grade
avg_by_grade = (
    df.groupBy("grade_int")
      .agg(function.round(function.avg("total_clean"), 2).alias("avg_total"))
      .orderBy(function.col("avg_total").desc())
)
avg_by_grade.show(10, truncate=False)

+---------+---------+
|grade_int|avg_total|
+---------+---------+
|12       |184.13   |
|10       |171.64   |
|1        |170.64   |
|3        |168.4    |
|6        |167.35   |
|7        |155.91   |
|5        |155.69   |
|11       |153.7    |
|8        |143.86   |
|9        |139.73   |
+---------+---------+
only showing top 10 rows


In [ ]:
# Insight 2: average subject scores by standardized gender
avg_by_gender = (
    df.groupBy("gender_std")
      .agg(
          function.round(function.avg("math_clean"), 2).alias("avg_math"),
          function.round(function.avg("science_clean"), 2).alias("avg_science"),
          function.round(function.avg("english_clean"), 2).alias("avg_english"),
          function.round(function.avg("total_clean"), 2).alias("avg_total")
      )
)
avg_by_gender.show(truncate=False)

+----------+--------+-----------+-----------+---------+
|gender_std|avg_math|avg_science|avg_english|avg_total|
+----------+--------+-----------+-----------+---------+
|Female    |53.55   |52.0       |50.95      |156.5    |
|Male      |52.58   |49.51      |52.09      |154.19   |
+----------+--------+-----------+-----------+---------+



In [ ]:
# Insight 3: top 10 students by total_clean
top_students = (
    df.select("name", "gender_std", "grade_int", "math_clean", "science_clean", "english_clean", "total_clean")
      .orderBy(function.col("total_clean").desc())
      .limit(10)
)
top_students.show(truncate=False)

+-------+----------+---------+----------+-------------+-------------+-----------+
|name   |gender_std|grade_int|math_clean|science_clean|english_clean|total_clean|
+-------+----------+---------+----------+-------------+-------------+-----------+
|Navya  |Male      |6        |94.0      |96.0         |94.0         |284.0      |
|Aditi  |Female    |1        |92.0      |90.0         |93.0         |275.0      |
|Myra   |Female    |5        |99.0      |95.0         |75.0         |269.0      |
|Aryan  |Male      |12       |79.0      |87.0         |98.0         |264.0      |
|Ishaan |Male      |7        |90.0      |97.0         |77.0         |264.0      |
|Isha   |Male      |3        |66.0      |90.0         |94.0         |250.0      |
|Reyansh|Male      |6        |100.0     |59.0         |89.0         |248.0      |
|Aryan  |Male      |1        |87.0      |61.0         |97.0         |245.0      |
|Aditi  |Male      |10       |93.0      |77.0         |74.0         |244.0      |
|Isha   |Female 

In [ ]:
# LOAD (DataFrameWriter, Parquet)
output_path = r"sample_data/student_data_clean.parquet"
(
    df.select(
        "name",
        "gender_std",
        "grade_int",
        "math_clean",
        "science_clean",
        "english_clean",
        "total_clean"
    )
    .write
    .mode("overwrite")
    .parquet(output_path)
)

In [ ]:
# Drop original raw columns and keep only cleaned ones
df_cleaned = (
    df
    .select(
        "name",
        "gender_std",
        "grade_int",
        "math_clean",
        "science_clean",
        "english_clean",
        "total_clean"
    )
)

In [ ]:
# Optionally print schema and a sample (printSchema, show)
df_cleaned.printSchema()
df_cleaned.show(20, truncate=False)

root
 |-- name: string (nullable = true)
 |-- gender_std: string (nullable = false)
 |-- grade_int: integer (nullable = true)
 |-- math_clean: double (nullable = false)
 |-- science_clean: double (nullable = false)
 |-- english_clean: double (nullable = false)
 |-- total_clean: double (nullable = false)

+-------+----------+---------+----------+-------------+-------------+-----------+
|name   |gender_std|grade_int|math_clean|science_clean|english_clean|total_clean|
+-------+----------+---------+----------+-------------+-------------+-----------+
|Advika |Female    |1        |84.0      |13.0         |53.0         |150.0      |
|Ishaan |Male      |3        |33.0      |47.0         |52.0         |132.0      |
|Zara   |Female    |7        |27.0      |14.0         |45.0         |86.0       |
|Myra   |Male      |1        |67.0      |35.0         |71.0         |173.0      |
|Advika |Female    |9        |23.0      |64.0         |92.0         |179.0      |
|Myra   |Male      |8        |35.0    